## 📦 Amazon Fine Food Reviews Dataset — Summary

**🔗 Source:** [Kaggle - Amazon Fine Food Reviews](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

### 📄 Columns in the dataset:
1. `Id` – Unique identifier for each review  
2. `ProductId` – Identifier for the product  
3. `UserId` – Identifier for the user  
4. `ProfileName` – Name of the reviewer  
5. `HelpfulnessNumerator` – Number of users who found the review helpful  
6. `HelpfulnessDenominator` – Number of users who evaluated the helpfulness  
7. `Score` – Rating (1 to 5)  
8. `Time` – Timestamp for the review (Unix format)  
9. `Summary` – Short summary of the review  
10. `Text` – Full text of the review

### 📝 Brief Summary:
This dataset contains **568,454 Amazon food reviews** spanning from **1999 to 2012**, including product metadata, user ratings, and review text. It's widely used for **sentiment analysis, NLP tasks, and recommender systems** research.


In [11]:
select top 10 * from Reviewsfood order by ID desc

(10 rows affected)

Total execution time: 00:00:01.554

Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,embedding
568454,B001LR2CU2,A3LGQPJCZVL9UC,srfell17,0,0,5,1338422400,Great Honey,"I am very satisfied ,product is as advertised, I use it on cereal, with raw vinegar, and as a general sweetner.",NULL
568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rewarding your dog for being good while grooming. Lower in calories and loved by all the doggies. Sweet potatoes seem to be their favorite Wet Noses treat!,NULL
568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 of those in one training session. I tried to train our dog with ""Ceaser dog treats"", it just made our puppy hyper. If you compare the ingredients, you will know why. Little stars has just basic food ingredients without any preservatives and food coloring. Sweet potato flavor also did not make my hand smell like dog food.",NULL
568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,"I'm disappointed with the flavor. The chocolate notes are especially weak. Milk thickens it but the flavor still disappoints. This was worth a try but I'll never buy again. I will use what's left, which will be gone in no time thanks to the small cans.",NULL
568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if not better than resturants I have eaten at..My husband loved it..will find other recipes to use this in..,NULL
568449,B001EO7N10,A1F6BHEYB7R6R7,James Braley,0,0,5,1308096000,Very large ground spice jars.,"My only complaint is that there's so much of it, I don't use a huge amount of 5 spice, so I gave 1 jar to my sister. Good enough.",NULL
568448,B001EO7N10,APWCOAVILK94B,"Real Named Person ""wowzee""",0,0,5,1322524800,"If its all natural, this is like panacea of Spice mixes","Hoping there is no MSG in this, this tastes extremely good when i make it with toasted fish. I used to eat fried chicken and other chicken dishes when i was young and had no idea they tasted so good because of this spice mix. Now i know when i remember about the great aroma and lip smacking flavor this imparts to my fish. This is also good in soups and gives you a kind of addiction that you will want to put this spice mix into everything you eat.<br />Would like to add this will make anyone eat any kind of meat or vegetable, in my case I prefer costly red salmon in a can but pink salmon is so cheaper and now with Five spice pink salmon toast feels as enjoyable as anything else i have eaten.",NULL
568447,B001EO7N10,A2P9W8T7NTLG2Z,Andy,0,0,2,1328918400,Mixed wrong,"I had ordered some of these a few months back and they were great, but the latest batch was terrible. All anise is what it tasted like. They sent two more that we're just as bad.<br /><br />Maybe it will eventually get the right mix back in stock but I ordered elsewhere.",NULL
568446,B001EO7N10,A2E5C8TTAED4CQ,S. Linkletter,2,2,5,1268006400,Five Spice Powder,"You can make this mix yourself, but the Star Anise is often difficult to find. Usually I use this mix to make Oriental-style pork or chicken dishes, but today I added a little to some fish boiled with lemon juice and black pepper. It was very good without being actually Oriental in overall impression.",NULL
568445,B001EO7N10,A2SD7TY3IOX69B,"BayBay ""BayBay Knows Best""",3,3,5,1245369600,Best Value for Chinese 5 Spice,"As a foodie, I use a lot of Chinese 5 Spice powder in my daily cooking. It is impossible to get the amount of 5 spice for this price! All my dishes taste great!",NULL


This script loops through review IDs from **1 to 568455** in the `Reviewsfood` table.  
It generates embeddings from the `Text` column using `dbo.get_embedding` and updates the `Embedding` column.


In [ ]:

    DECLARE @currentId INT = 9301;
    DECLARE @endId INT = 568455;
    DECLARE @text NVARCHAR(MAX);
    DECLARE @embedding VARBINARY(MAX);

    WHILE @currentId <= @endId
    BEGIN
        -- Get the text for the current row
        SELECT @text = Text
        FROM Reviewsfood
        WHERE Id = @currentId;

        -- Generate the embedding
        EXEC dbo.get_embedding 
            @deployedModelName = 'text-embedding-3-small',
            @inputText = @text,
            @embedding = @embedding OUTPUT;

        -- Update the embedding column
        UPDATE Reviewsfood
        SET Embedding = @embedding
        WHERE Id = @currentId;

        -- Move to the next row
        SET @currentId = @currentId + 1;
    END

## 🔍 Semantic Vector Search in T-SQL

This script performs a **semantic search** over product reviews using vector embeddings:

1. **Embedding Generation**  
   It uses the `dbo.get_embedding` stored procedure to convert the input query `'oatmeal options for my toddler'` into a 1536-dimensional vector.

2. **Similarity Search**  
   It compares this query embedding with the `Embedding` column in the `Reviewsfood` table  
   using `vector_distance('cosine', ...)` to measure semantic similarity.

3. **Top-N Results**  
   It retrieves the top 4 most similar reviews based on cosine similarity, returning:  
   - `ProductId`  
   - `Summary`  
   - `Text`  
   - `SimilarityScore`

This approach enables **natural language search** over unstructured text using vector-based matching.


In [18]:
DECLARE @query NVARCHAR(MAX) = 'oatmeal options for my toddler';
DECLARE @embedding vector(1536);
DECLARE @num_results INT = 4;

-- Step 1: Generate embedding for the input query
EXEC dbo.get_embedding 
    @deployedModelName = 'text-embedding-3-small',
    @inputText = @query,
    @embedding = @embedding OUTPUT;

-- Step 2: Search for top N most similar reviews
SELECT TOP (@num_results)
    ProductId,
    Summary,
    Text,
    vector_distance('cosine', Embedding, @embedding) AS SimilarityScore
FROM Reviewsfood
WHERE Embedding IS NOT NULL
ORDER BY SimilarityScore ASC;


(4 rows affected)

Total execution time: 00:00:01.120

ProductId,Summary,Text,SimilarityScore
B001DIM8K8,Texture and the taste are totally different from the pressed oats!,I have been having Quaker Oatmeal for yesrs until trying this today! The texture and the taste are completely different from the pressed oat. It worths cooking for half hour. My son love it at the first bite! My 8-month-old baby is also love it! I uaually cook the portion which will be enough for 2~3 days. Drop one spoon or two into the worm milk every morning to make my breakfast much more healthier! This one is so good that it will be hard for me to go back to pressed oatmeal.....,0.39622429188824226
B001DIM8K8,Slow cooking,"I'll be honest, if you want a quick breakfast don't purchase this oatmeal. But, if you have time to spare or don't mind waiting for some ""good eats"", this is the oatmeal for you.<br /><br />Step 1. pour some milk into a sauce pan; Step 2. add a dash of salt [or salt to taste]; Step 3. add oatmeal, making sure it stays covered by milk; Step 4. turn on stove to low heat [not too low but definitely not too high, as you don't want the milk to evaporate/dry out leaving you with tough oats in pan] and cover pan; Step 5. do some little cleaning to built up appetite making sure to ck on progress periodically; Step 6. add more milk if oats are still too tough and let it simmer for a little while longer; Step 7. get your ladle/just pour directing into a bowl add a few raisins/cranberries/your favorite dried fruit and ENJOY!",0.42752723030117457
B001DIM8K8,Nothing like the crappy stuff I grew up on.,"My daughter and I eat low-carb. Which oatmeal is not really, by the way. But I was looking for something that we could eat in the morning that was not meat/dairy/egg, and was gluten-free, and filling, without being pure sugar like most breakfast foods and especially grains. I decided to try these long-cooking irish oats. We use 2 cups water and 1/2 cup dry oats, for 1/4cup dry each. It takes this stuff 30 minutes+ to cook so it is not fast food, for sure. But strangely enough, it's filling. Adding fats (couple tablespoons of cream) to the serving helps that of course. In any case, it is much tastier, and much more long-lasting for keeping us satisfied, than I expected, and surprisingly stable on blood sugar. Nor has it set off cravings like most grains, fruits, etc. do if we eat those; the sugars digest so slowly in this apparently. It is more solid and nutty than the flakey quick-oats I grew up with, which I now consider a gummy gluey sugarbomb insult to true oats. These are harder to find and cost more but they're worth it.",0.4495868671558576
B001EO5QW8,it's oatmeal,"What else do you need to know? Oatmeal, instant (make it with a half cup of low-fat milk and add raisins;nuke for 90 seconds). More expensive than Kroger store brand oatmeal and maybe a little tastier or better texture or something. It's still just oatmeal. Mmm, convenient!",0.45345380292082305
